# KPI pregled

Učitava `outputs/evaluations/kpi_by_level.csv` i prikazuje dva uporedna pregleda:
1. **Modeli po nivou** — jedna tabela po nivou hijerarhije, modeli rangirani po WMAPE.
2. **Nivoi po modelu** — jedna tabela po modelu, nivoi rangirani po WMAPE.

Sve metrike su skalno-nezavisne (procentualne ili bezdimenzione), tako da se vrednosti mogu direktno porediti između nivoa hijerarhije.

## Korišćene metrike

Oznake: $a_t$ = stvarna vrednost, $f_t$ = prognozirana vrednost, $n$ = broj posmatranja, $e_t = f_t - a_t$ (greška sa znakom).

### WMAPE — Weighted Mean Absolute Percentage Error
$$\text{WMAPE} = \frac{\sum_{t=1}^{n} |a_t - f_t|}{\sum_{t=1}^{n} |a_t|} \times 100$$
Meri **veličinu** greške, normalizovano ukupnim obimom. Manje je bolje; 0 = savršeno. Otporno na nule (za razliku od MAPE).

### nRMSE — Normalized Root Mean Squared Error
$$\text{nRMSE} = \frac{\sqrt{\frac{1}{n}\sum_{t=1}^{n} (a_t - f_t)^2}}{\frac{1}{n}\sum_{t=1}^{n} |a_t|} \times 100$$
RMSE normalizovan srednjom apsolutnom vrednošću aktuala. Više kažnjava velike greške nego WMAPE (kvadratni član), pa je osetljiviji na izuzetke.

### MdAPE — Median Absolute Percentage Error
$$\text{MdAPE} = \text{median}_{t : |a_t| > \epsilon}\left( \frac{|a_t - f_t|}{|a_t|} \right) \times 100$$
Medijana procentualnih grešaka po tačkama. Robustna na izuzetke; ignoriše redove sa $|a_t| \approx 0$ koje imaju vrlo negativan uticaj i pri relativno malim greškama zbog deljenja sa EPS vrednoscu(~0,00001).

### sMAPE — Symmetric Mean Absolute Percentage Error
$$\text{sMAPE} = \frac{1}{n} \sum_{t=1}^{n} \frac{2 \, |a_t - f_t|}{|a_t| + |f_t| + \epsilon} \times 100$$
Simetrična varijanta MAPE: jednako tretira pre- i pod-prognoziranje. Ograničena na $[0, 200\%]$, definisana i kada je $a_t = 0$.

### BIAS_PCT — Normalized Bias
$$\text{BIAS\_PCT} = \frac{\frac{1}{n}\sum_{t=1}^{n} (f_t - a_t)}{\frac{1}{n}\sum_{t=1}^{n} |a_t|} \times 100 = \frac{\sum (f_t - a_t)}{\sum |a_t|} \times 100$$
Meri **smer** sistemske greške:
- $> 0$ → model sistematski **precenjuje** (over-forecast),
- $< 0$ → model sistematski **potcenjuje** (under-forecast),
- $\approx 0$ → nepristrasan.

### TS — Tracking Signal
$$\text{TS} = \frac{\sum_{t=1}^{n} (f_t - a_t)}{\text{MAD}}, \quad \text{MAD} = \frac{1}{n}\sum_{t=1}^{n} |f_t - a_t|$$
Kontrolno-kartni indikator pristrasnosti. Često korišćena granica: $|TS| > 4$ ukazuje na sistematsku pristrasnost koju treba istražiti. Vrednost bliska 0 = uravnotežene greške.

## Pravila isticanja u tabelama

| Metrika | Najbolje |
|---|---|
| WMAPE, nRMSE, MdAPE, sMAPE | minimalna vrednost |
| BIAS_PCT, TS | vrednost najbliža nuli ($|x|$ minimalno) |

In [41]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
KPI_PATH = REPO_ROOT / 'outputs' / 'evaluations' / 'kpi_by_level.csv'

kpi = pd.read_csv(KPI_PATH)
print(f'Loaded {len(kpi)} rows from {KPI_PATH.relative_to(REPO_ROOT)}')
print(f"Levels: {kpi['level'].nunique()}  |  Models: {kpi['model'].nunique()}")

Loaded 176 rows from outputs\evaluations\kpi_by_level.csv
Levels: 16  |  Models: 11


In [42]:
# Konfigurabilne opcije prikaza
METRICS = ['WMAPE', 'nRMSE', 'MdAPE', 'sMAPE', 'BIAS_PCT', 'TS']
LOWER_IS_BETTER = ['WMAPE', 'nRMSE', 'MdAPE', 'sMAPE']   # najmanja vrednost je najbolja
CLOSEST_TO_ZERO = ['BIAS_PCT', 'TS']                      # vrednost najbliža nuli je najbolja
RANK_BY = 'WMAPE'   # kolona po kojoj se sortira unutar svake tabele
ASCENDING = True

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pd.set_option('display.max_rows', 200)

## 1. Modeli po nivou

Za svaki nivo hijerarhije, prikazani su svi modeli jedan pored drugog. Najbolja vrednost svake metrike je istaknuta zelenom bojom.

In [43]:
from IPython.display import display, Markdown

GREEN = 'background-color:#c8e6c9;color:black;'


def _closest_to_zero(col: pd.Series) -> list:
    """Return a list of CSS strings, marking the row whose |value| is smallest."""
    abs_col = col.abs()
    if abs_col.dropna().empty:
        return ['' for _ in col]
    best_idx = abs_col.idxmin()
    return [GREEN if i == best_idx else '' for i in col.index]


def _style_table(table: pd.DataFrame) -> pd.io.formats.style.Styler:
    styled = table.style.format({m: '{:,.3f}' for m in METRICS}).hide(axis='index')
    lower_cols = [c for c in LOWER_IS_BETTER if c in table.columns]
    zero_cols = [c for c in CLOSEST_TO_ZERO if c in table.columns]
    if lower_cols:
        styled = styled.highlight_min(subset=lower_cols, props=GREEN)
    if zero_cols:
        styled = styled.apply(_closest_to_zero, subset=zero_cols, axis=0)
    return styled


for level, group in kpi.groupby('level', sort=False):
    table = (
        group[['model'] + METRICS]
        .sort_values(RANK_BY, ascending=ASCENDING)
        .reset_index(drop=True)
    )
    display(Markdown(f'### {level}'))
    display(_style_table(table))

### base

model,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
deepar,110.168,142.358,71.362,107.997,-4.165,-2.187
nhits,113.382,143.207,71.361,116.331,6.533,-1.649
timesfm,118.560,147.787,70.819,112.012,15.398,-1.279
chronos,126.325,157.920,76.964,118.185,20.896,-1.318
holt_winters,188.967,235.011,96.723,103.624,104.753,0.593
prophet,190.839,228.195,93.690,103.862,103.814,0.238
arima,192.391,217.582,85.006,106.531,120.403,0.908
random_forest,301.592,318.607,111.993,106.751,249.863,1.861
random_forest_direct,304.113,322.923,109.190,106.264,256.082,2.134
lightgbm,865.400,878.490,336.374,120.284,832.730,3.154


### structural__customer

model,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
deepar,65.711,80.194,52.928,64.678,1.786,-1.097
timesfm,88.948,102.620,61.168,66.845,28.813,-0.704
chronos,100.866,114.344,64.537,66.787,44.197,-0.308
nhits,112.067,138.726,85.539,73.398,51.967,-0.159
holt_winters,117.767,143.460,93.902,71.259,67.679,1.080
prophet,129.422,144.662,98.020,68.745,83.285,1.011
arima,140.121,152.016,100.675,65.471,103.207,1.418
random_forest,212.497,227.448,117.790,66.478,176.632,1.676
random_forest_direct,257.257,295.404,102.109,66.682,220.064,1.555
lightgbm_direct,438.149,450.610,172.672,68.704,400.105,1.457


### structural__location

...

model,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
deepar,58.838,64.553,64.625,48.238,16.157,-0.375
chronos,63.346,68.814,70.704,47.921,25.535,-0.064
timesfm,66.223,71.366,72.189,49.181,27.616,-0.040
nhits,66.258,71.479,75.371,46.662,33.672,0.090
holt_winters,78.116,86.261,94.606,52.897,41.742,0.188
prophet,79.109,84.911,93.030,53.320,44.311,0.219
arima,84.835,92.497,94.948,50.989,54.465,0.427
random_forest,103.128,108.083,111.507,49.969,80.222,0.560
lightgbm_direct,106.553,113.911,120.841,54.877,80.155,0.531
lightgbm,108.854,122.627,127.905,57.919,77.856,0.508


### temporal__quarter__structural__PH3

model,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
chronos,62.295,67.926,67.234,51.466,21.649,-0.027
deepar,63.455,68.649,68.096,50.859,24.758,0.037
timesfm,65.132,70.627,68.908,52.446,24.758,0.018
nhits,67.149,72.555,70.439,49.940,28.283,-0.077
prophet,82.382,88.273,89.668,56.514,43.927,0.183
holt_winters,83.062,91.347,92.217,57.669,41.500,0.128
arima,90.255,101.701,92.740,55.913,55.647,0.366
random_forest,109.912,115.032,113.131,53.221,88.206,0.693
random_forest_direct,112.724,118.432,113.306,52.489,89.557,0.569
lightgbm_direct,126.452,135.872,127.603,56.005,102.297,0.588


### temporal__quarter__structural__total

model,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
lightgbm_direct,0.711,0.839,0.710,0.714,-0.711,-2.000
random_forest_direct,0.754,0.852,0.754,0.757,-0.754,-2.000
chronos,0.791,0.905,0.790,0.786,0.791,2.000
random_forest,0.884,0.905,0.884,0.888,-0.884,-2.000
lightgbm,0.892,0.906,0.891,0.896,-0.892,-2.000
arima,1.795,2.406,1.797,1.827,-1.795,-2.000
timesfm,3.585,3.592,3.585,3.651,-3.585,-2.000
deepar,4.078,4.308,4.080,4.026,1.388,0.680
nhits,8.375,8.618,8.372,8.220,2.031,0.485
prophet,9.061,10.209,9.069,8.578,9.061,2.000


## 2. Nivoi po modelu

Za svaki model, prikazani su svi nivoi jedan pored drugog. Najbolja vrednost svake metrike je istaknuta zelenom bojom.

In [45]:
for model, group in sorted(kpi.groupby('model', sort=False), key=lambda kv: kv[0]):
    table = (
        group[['level'] + METRICS]
        .sort_values('level')
        .reset_index(drop=True)
    )
    display(Markdown(f'### {model}'))
    display(_style_table(table))

### arima

level,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
base,192.391,217.582,85.006,106.531,120.403,0.908
structural__CBF,59.190,65.003,723.760,47.114,26.037,0.506
structural__PH1,78.138,88.099,80.207,57.649,38.498,0.747
structural__PH2,104.776,119.291,83.517,70.457,56.381,0.883
structural__PH3,104.693,120.163,84.558,74.800,54.291,0.962
structural__customer,140.121,152.016,100.675,65.471,103.207,1.418
structural__location,37.246,44.842,31.359,46.621,-12.720,-0.605
structural__total,9.159,11.131,7.919,9.170,-4.795,-3.141
temporal__quarter__base,191.401,205.560,183.935,80.523,143.973,0.350
temporal__quarter__structural__CBF,108.988,137.111,696.390,31.361,92.032,0.006


### chronos

level,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
base,126.325,157.920,76.964,118.185,20.896,-1.318
structural__CBF,44.683,55.070,353.446,46.683,2.180,0.152
structural__PH1,51.230,63.735,48.600,54.871,0.770,-0.542
structural__PH2,68.979,86.860,58.130,70.657,6.320,-0.522
structural__PH3,69.133,88.239,59.492,76.419,-0.150,-0.725
structural__customer,100.866,114.344,64.537,66.787,44.197,-0.308
structural__location,37.768,50.084,34.587,51.646,-17.503,-0.728
structural__total,8.879,10.204,8.405,8.816,4.511,3.048
temporal__quarter__base,108.868,116.552,105.405,77.889,51.322,-0.067
temporal__quarter__structural__CBF,24.595,29.434,454.303,24.497,3.479,-0.222


### deepar

...

level,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
base,301.592,318.607,111.993,106.751,249.863,1.861
structural__CBF,62.397,70.449,400.264,50.048,23.907,-0.600
structural__PH1,69.712,79.433,70.899,56.432,33.769,1.001
structural__PH2,530.623,540.948,314.086,79.273,501.201,2.224
structural__PH3,107.237,120.637,80.923,72.811,63.114,1.396
structural__customer,212.497,227.448,117.790,66.478,176.632,1.676
structural__location,323.096,412.782,439.075,49.494,293.275,0.848
structural__total,8.507,10.556,5.549,8.503,-2.598,-1.832
temporal__quarter__base,333.102,342.336,276.517,81.482,309.747,0.894
temporal__quarter__structural__CBF,90.640,95.584,518.920,33.642,78.810,0.655


### random_forest_direct

level,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
base,304.113,322.923,109.190,106.264,256.082,2.134
structural__CBF,150.003,197.488,522.094,52.029,129.488,2.336
structural__PH1,92.702,108.859,82.945,59.230,62.626,1.893
structural__PH2,321.533,428.833,141.667,72.219,287.193,2.054
structural__PH3,102.652,116.674,74.967,72.566,54.981,0.964
structural__customer,257.257,295.404,102.109,66.682,220.064,1.555
structural__location,55.580,72.232,74.776,47.092,15.192,-0.332
structural__total,7.945,9.310,6.503,7.924,-4.301,-3.248
temporal__quarter__base,249.341,259.747,214.736,78.690,221.887,0.745
temporal__quarter__structural__CBF,95.480,101.373,505.889,33.796,80.896,0.559


### timesfm

level,WMAPE,nRMSE,MdAPE,sMAPE,BIAS_PCT,TS
base,118.560,147.787,70.819,112.012,15.398,-1.279
structural__CBF,42.920,51.259,383.683,44.938,-2.969,-1.231
structural__PH1,50.299,61.977,47.270,55.686,-4.965,-0.908
structural__PH2,74.547,91.680,57.601,68.366,9.051,-0.909
structural__PH3,73.788,92.415,59.053,74.561,3.180,-0.950
structural__customer,88.948,102.620,61.168,66.845,28.813,-0.704
structural__location,37.330,49.407,33.442,51.544,-23.169,-2.300
structural__total,7.789,9.300,8.985,7.756,0.428,0.330
temporal__quarter__base,116.197,123.749,107.621,77.506,60.196,-0.040
temporal__quarter__structural__CBF,31.463,35.881,547.359,28.815,5.775,-0.755
